# 01. Загрузка данных IMDb в PostgreSQL

## Цель
Загрузить 7 TSV-файлов датасета IMDb в PostgreSQL
с использованием команды `COPY` (потоковая загрузка).

## Источник данных
- [IMDb Non-Commercial Datasets](https://datasets.imdbws.com/)
- Формат: TSV, UTF-8, разделитель `\t`, NULL = `\N`

In [1]:
import os
from pathlib import Path
import pandas as pd
from sqlalchemy import create_engine, text, inspect
from dotenv import load_dotenv
import time 

load_dotenv(Path("../.env"))
DB_URL = (
    f"postgresql+psycopg://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)
engine = create_engine(DB_URL)

## Очистка таблиц  через `TRUNCATE`


In [ ]:
#   tables = inspect(engine).get_table_names(schema="imdb")
#   with engine.connect() as conn:
#    for table in tables:
#        conn.execute(text(f"TRUNCATE TABLE imdb.{table} CASCADE" ))
#        print(f"{table} cleared")
    

## Загрузка в базу данных

### Порядок загрузки на основе **FK**

In [2]:
DATA_DIR = Path("../data/raw")

#Порядок FK зависимостей
load_order = [
    ("name.basics.tsv",     "name_basics"),
    ("title.basics.tsv",    "title_basics"),
    ("title.ratings.tsv",   "title_ratings"),
    ("title.crew.tsv",      "title_crew"),
    ("title.episode.tsv",   "title_episode"),
    ("title.principals.tsv","title_principals"),
    ("title.akas.tsv",      "title_akas"),
]

for file_name, table_name in load_order:
    file_path = DATA_DIR / file_name
    status = "ФаЙЛ найден" if file_path.exists() else "Файл не найден"
    print(f"{status}  {file_name}")

Файл не найден  name.basics.tsv
Файл не найден  title.basics.tsv
Файл не найден  title.ratings.tsv
Файл не найден  title.crew.tsv
Файл не найден  title.episode.tsv
Файл не найден  title.principals.tsv
Файл не найден  title.akas.tsv


### Функция загрузки 

In [3]:
def table_copy(file_name, table_name):

    file_path = DATA_DIR / file_name
    con = engine.raw_connection()

    try:
        with con.cursor() as cursor:
            with open(file_path, "rb") as f:
                with cursor.copy(
                    f"""
                    COPY imdb.{table_name}
                    FROM STDIN
                    WITH (
                        FORMAT text,
                        DELIMITER E'\\t',
                        HEADER true,
                        NULL '\\N'
                    )
                    """
                ) as copy:

                    while data := f.read(1024 * 1024):
                        copy.write(data)

        con.commit()

        print(f"{table_name} loaded")

    except Exception:
        con.rollback()
        raise

    finally:
        con.close()

### Загрузка данных 

In [ ]:
for file_name, table_name in load_order:
       table_copy(file_name, table_name)   

## Проверка

### Функция проверки

In [5]:
def check_load(file_name, table_name):

    file_path = DATA_DIR / file_name

    # Количество строк в файле
    with open(file_path, "rt", encoding="utf-8") as f:
        file_rows = sum(1 for i in f) - 1
    
    # Количество строк в БД
    with engine.connect() as conn:
        db_rows = conn.execute(
            text(f"SELECT COUNT(*) FROM imdb.{table_name}")
        ).scalar()
    
    difference = file_rows - db_rows

    return {
        "Таблица " : table_name,
        "Файл" :    file_name,
        "Количество строк в БД": db_rows,
        "Разница количества строк": difference,
    }

### Проверка (количество строк должно совпадать)

In [ ]:
result = []
for file_name, table_name in load_order:
       result.append(check_load(file_name, table_name))
comparison = pd.DataFrame(result)
display(comparison)

## Сумма записей в БД

In [28]:
tables = inspect(engine).get_table_names(schema="imdb")
total_row = 0
with engine.connect() as con:
    for table in tables:
        rows = con.execute(
            text(f"SELECT COUNT(*) FROM imdb.{table}")
            ).scalar()
        total_row += rows
        print('Таблица: ', table, 'Количество: ', rows)
print('Общая сумма =', total_row)

Таблица:  genres Количество:  28
Таблица:  title_basics Количество:  12812871
Таблица:  title_genres Количество:  19973102
Таблица:  title_principals Количество:  101902684
Таблица:  title_episode Количество:  9907631
Таблица:  title_akas Количество:  59478714
Таблица:  name_basics Количество:  15681293
Таблица:  title_ratings Количество:  1713431
Таблица:  title_crew Количество:  12812871
Общая сумма = 234282625
